# A real decoder rollout → verifier → backward → update

Day 23 · Chapter 13 · CPU worked notebook

Prerequisites: Chapter5 decoder and Chapter13 objective. This is an actual CPU Transformer update, using original arithmetic tokens, not language-model benchmark data.

Predict: if every sampled answer in a group succeeds, what remains of the relative policy signal? Will increasing G necessarily improve four held-out prompts?

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: decoder rlvr](../figures/chapter-13/day-23-01_decoder_rlvr-01.png)

![Saved reference 2: decoder rlvr](../figures/chapter-13/day-23-01_decoder_rlvr-02.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.grpo_lab import decoder_rlvr
result = decoder_rlvr(group_size=4, updates=12, seed=2223)
history = result["history"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
steps = [r["update"] for r in history]
for ax, key, label in zip(axes, ["mean_reward", "exact_kl_nats", "zero_variance_fraction"],
                         ["Mean sampled verifier reward", "Exact reference KL (nats)", "Constant-group fraction"]):
    ax.plot(steps, [r[key] for r in history], marker="o")
    ax.set(xlabel="Fresh optimizer update", ylabel=label)
fig.tight_layout()
plt.show()
print("Initial held-out:", result["initial"]["heldout"])
print("Final held-out (all rows):", result["final"]["heldout"])

Reading guide: reward belongs to sampled training rollouts, KL is exact over vocabulary at those visited states, and constant-group fraction diagnoses relative signal availability. None is itself the held-out score.

Exercise: hold the initialization, prompt schedule and update count fixed, change G4 to G8, and compare quality and actual valid response-token cost.

In [ ]:
# Reference solution; unequal sample budget is explicitly part of this comparison.
larger = decoder_rlvr(group_size=8, updates=12, seed=2223)
for r in (result, larger):
    print({"G": r["group_size"],
           "response_tokens": sum(x["valid_response_tokens"] for x in r["history"]),
           "train_greedy": r["final"]["train"]["accuracy"],
           "heldout_greedy": r["final"]["heldout"]["accuracy"],
           "nonzero_advantage_group_updates": sum(x["zero_variance_fraction"] < 1 for x in r["history"])})
# Architecture/orchestration schematic; boxes represent operations, not measured quality.
fig, ax = plt.subplots(figsize=(11, 2.1))
labels = ["Frozen behavior\nrollout [B·G,T]", "Strict verifier\nrewards [B,G]",
          "Detached group\nadvantages [B,G]", "Current decoder\nlogits [B·G,T,V]", "Masked loss\nbackward + update"]
for i, label in enumerate(labels):
    ax.text(i, .5, label, ha="center", va="center",
            bbox={"facecolor":"white", "edgecolor":"#bbbbbb", "boxstyle":"square,pad=.5"})
    if i < len(labels)-1:
        ax.annotate("", xy=(i+.67,.5), xytext=(i+.35,.5), arrowprops={"arrowstyle":"->"})
ax.set(xlim=(-.6,4.6), ylim=(0,1))
ax.axis("off")
plt.show()

Interpretation: report failures, zero-variance groups and unequal compute. A larger G is an exploration/cost choice, not an automatic win. Response EOS counts as an action; later padding is masked. Current logits are recomputed with one-position causal shifting; behavior and reference are frozen.

Controlled change: compare normalized versus centered-only advantages with a fresh seed-matched run. Evidence boundary: one tiny arithmetic decoder and four held-out prompts; no Qwen or general reasoning result.